# DSPy Tour: signatures, modules, evaluation, optimization

A walk through the five core abstractions of DSPy — Language Models, Signatures, Modules, Evaluation, and Optimizers — following Chapter 2 of *Context Engineering with DSPy*.

Set `OPENAI_API_KEY` in a `.env` file at the repo root before running. The provider-switch section also uses `ANTHROPIC_API_KEY` and `GOOGLE_API_KEY` if they are set, and skips gracefully otherwise.


In [ ]:
%pip install -r ../requirements.txt -q

## Setup

Load environment variables, then configure DSPy globally with an OpenAI model. The `temperature=1` and `max_tokens=16000` settings are required by GPT-5 reasoning models.

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

lm = dspy.LM(
    'openai/gpt-5-mini',
    api_key=os.getenv('OPENAI_API_KEY'),
    temperature=1,
    max_tokens=16000,
)
dspy.configure(lm=lm)

## Setting up an LM

Once configured globally, you can call the LM directly to confirm everything works.

In [ ]:
lm("Hello")

## Changing LM Provider

DSPy's unified `LM` class lets you swap providers with a single line. No prompt rewriting, no format changes. The cell below sets up Anthropic and Gemini variants, skipping each one gracefully if the relevant key is missing.

In [ ]:
claude_lm = None
gem_lm = None

if os.getenv('ANTHROPIC_API_KEY'):
    claude_lm = dspy.LM(
        'anthropic/claude-haiku-4-5-20251001',
        api_key=os.getenv('ANTHROPIC_API_KEY'),
    )
    print('Anthropic LM ready:', claude_lm.model)
else:
    print('Skipping Anthropic: ANTHROPIC_API_KEY not set')

if os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY'):
    gem_lm = dspy.LM(
        'gemini/gemini-2.5-flash',
        api_key=os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'),
    )
    print('Gemini LM ready:', gem_lm.model)
else:
    print('Skipping Gemini: GOOGLE_API_KEY / GEMINI_API_KEY not set')

## Defining a Signature

Signatures replace brittle prompt strings with Python classes that declare inputs and outputs. The minimal form just lists fields; the richer form adds a docstring, type annotations, and field descriptions, all of which DSPy folds into the generated prompt.

In [ ]:
# Minimal signature
class QASignature(dspy.Signature):
    question = dspy.InputField()
    answer = dspy.OutputField()

In [ ]:
# Signature with docstring, types, and field descriptions
class QASignature(dspy.Signature):
    """Answer questions based on provided context with reasoning and confidence."""
    context: list[str] = dspy.InputField(desc="Relevant context retrieved from our database")
    question = dspy.InputField()
    reasoning = dspy.OutputField()
    answer = dspy.OutputField()
    confidence: float = dspy.OutputField()

DSPy signatures also support Pydantic models as custom output types, so the model returns a structured object that gets validated for you.

In [ ]:
from pydantic import BaseModel, Field

class AnswerWithConfidence(BaseModel):
    answer: str = Field(description="Clear, concise answer to the question")
    confidence: float = Field(
        description="Confidence score between 0 and 1",
        ge=0.0,
        le=1.0,
    )

class QASignaturePydantic(dspy.Signature):
    """Answer questions based on provided context with reasoning and confidence."""
    context: list[str] = dspy.InputField(desc="Relevant context retrieved from our database")
    question: str = dspy.InputField()
    reasoning: str = dspy.OutputField()
    output: AnswerWithConfidence = dspy.OutputField()

## Predict Module

`dspy.Predict` is the simplest module. Pass it a signature and call it like a function — DSPy builds the prompt, calls the LM, and parses the structured response.

In [ ]:
# Re-define the simple QA signature for the module examples
class QASignature(dspy.Signature):
    question = dspy.InputField()
    answer = dspy.OutputField()

module = dspy.Predict(QASignature)

result = module(question="What is the capital of France?")
print(result.answer)

## Built-In Modules

DSPy ships with modules that wrap common prompt-engineering patterns. `ChainOfThought` adds a reasoning field automatically — same signature, more reliable output.

In [ ]:
cot_module = dspy.ChainOfThought(QASignature)

cot_result = cot_module(question="What is 2 x 450?")

print(cot_result.reasoning)
print(cot_result.answer)

Under the hood, `ChainOfThought` is just `Predict` with an extra `reasoning` output field. You could write it manually like this.

In [ ]:
class QAWithCoTSignature(dspy.Signature):
    question = dspy.InputField(desc="The question to answer.")
    reasoning = dspy.OutputField()
    answer = dspy.OutputField(desc="The final answer.")

manual_cot_module = dspy.Predict(QAWithCoTSignature)
manual_cot_result = manual_cot_module(question="What is 2 x 450?")
print(manual_cot_result.reasoning)
print(manual_cot_result.answer)

## ReAct Agents

`dspy.ReAct` implements the Reason+Act loop for tool-calling agents. Define your tools as plain Python functions, pass them in, and DSPy handles the thought → action → observation cycle.

In [ ]:
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # In a real implementation, this would call a weather API
    return f"The weather in {city} is sunny and 75F"

react_agent = dspy.ReAct(
    signature=QASignature,
    tools=[get_weather],
    max_iters=5,
)

result = react_agent(question="What's the weather like in Tokyo?")
print(result.answer)
print("Tool calls made:", result.trajectory)

## Example Datasets

DSPy evaluation and optimization both operate over `dspy.Example` objects. `.with_inputs(...)` tells DSPy which fields are program inputs and which are expected outputs.

In [ ]:
examples = [
    dspy.Example(
        question="What is the capital of France?",
        context="France is a country in Western Europe. Its capital is Paris, which is known for its art, fashion, gastronomy and culture.",
        answer="Paris",
    ).with_inputs("question", "context"),
    dspy.Example(
        question="What is the speed of light?",
        context="The speed of light in a vacuum is approximately 299,792 kilometers per second, often rounded to 300,000 km/s.",
        answer="300,000 km/s",
    ).with_inputs("question", "context"),
]

print(f"{len(examples)} examples ready")

You can also keep datasets as plain dicts and convert them with a list comprehension — easier to read and edit at scale.

In [ ]:
dataset = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Western Europe. Its capital is Paris, which is known for its art, fashion, gastronomy and culture.",
        "answer": "Paris",
    },
    {
        "question": "What is the speed of light?",
        "context": "The speed of light in a vacuum is approximately 299,792 kilometers per second, often rounded to 300,000 km/s.",
        "answer": "300,000 km/s",
    },
]

examples = [
    dspy.Example(
        question=item["question"],
        context=item["context"],
        answer=item["answer"],
    ).with_inputs("question", "context")
    for item in dataset
]

print(f"{len(examples)} examples ready")

## Evaluation Metrics

A metric is just a function that compares an expected example to a prediction and returns a score. The simplest is exact match.

In [ ]:
def exact_match(example, prediction, trace=None):
    """Check if prediction exactly matches expected output."""
    return example.answer == prediction.answer

Numeric metrics give optimizers finer-grained feedback. The `trace` parameter lets the same function return a boolean for optimizers that need a pass/fail signal.

In [ ]:
def word_count(example, pred, trace=None):
    """Returns a float between 0.0 and 1.0 based on word overlap, or boolean if trace is provided."""
    gold_words = example.answer.lower().split()
    pred_words = set(pred.answer.lower().split())

    if len(gold_words) == 0:
        return False if trace is not None else 0.0

    matching_words = sum(1 for word in gold_words if word in pred_words)
    score = matching_words / len(gold_words)

    if trace is not None:
        return score > 0.75

    return score

## Running Evaluations

Before optimizing, establish a baseline. You can loop through your dataset manually...

In [ ]:
program = dspy.Predict(QASignature)

scores = []
for example in examples:
    result = program(question=example.question)
    score = exact_match(example, result)
    scores.append(score)

print(f"Average score: {sum(scores) / len(scores)}")

...or use `dspy.Evaluate`, which runs your program in parallel, shows progress, and prints a results table.

In [ ]:
evaluator = dspy.Evaluate(
    devset=examples,
    metric=exact_match,
    num_threads=4,
    display_progress=True,
    display_table=5,
)

score = evaluator(program)
print(f"Accuracy: {score}%")

## Train-Test Split

Splitting your data into train, validation, and test sets protects against overfitting. DSPy tends to favor unusually large validation sets because LM optimizers learn from fewer examples than traditional ML algorithms.

In [ ]:
import random

# For reproducibility
random.seed(42)

# For fairness
random.shuffle(examples)

# Split: 20% train, 40% val, 40% test
n = len(examples)
n_train = max(1, int(0.2 * n))
n_val = max(1, int(0.4 * n))

trainset = examples[:n_train]
valset = examples[n_train:n_train + n_val]
testset = examples[n_train + n_val:]

print("Train set:", len(trainset))
print("Val set:", len(valset))
print("Test set:", len(testset))

## BootstrapFewShot Optimizer

`BootstrapFewShot` runs your program on the training set, keeps the outputs that pass your metric, and bakes them into the prompt as few-shot demonstrations. Good for trainsets in the 10-50 example range.

In [ ]:
from dspy.teleprompt import BootstrapFewShot

# 1. Create an optimizer
optimizer = BootstrapFewShot(
    metric=exact_match,
    max_bootstrapped_demos=4,  # generate new examples
    max_labeled_demos=8,       # add from your dataset
)

# 2. Compile your program
optimized_program = optimizer.compile(
    program,
    trainset=trainset,
)

# 3. Use your optimized program
result = optimized_program(question="What is context engineering?")
print(result.answer)

Inspect the optimized program's prompt to see the bootstrapped demonstrations DSPy added.

In [ ]:
dspy.inspect_history(n=1)

## GEPA Optimizer

GEPA (Genetic Pareto) actually rewrites the prompt instead of just adding demonstrations. It uses an evolutionary loop guided by a reflection LM — typically a larger, smarter model — that reads your evaluation results and proposes prompt mutations.

The cell below uses `auto='light'` as a smoke-test budget cap. The full-budget version is shown commented out alongside.

In [ ]:
# Smarter LM used for reflection (reads failures, proposes prompt rewrites)
teacher_lm = dspy.LM('openai/gpt-5', temperature=1.0, max_tokens=32000)

# Smoke-test budget cap
gepa_optimizer = dspy.GEPA(
    metric=exact_match,
    auto='light',
    num_threads=4,
    track_stats=True,
    reflection_lm=teacher_lm,
)

# Full-budget version (uncomment to run for real):
# gepa_optimizer = dspy.GEPA(
#     metric=exact_match,
#     max_full_evals=3,
#     num_threads=16,
#     track_stats=True,
#     use_merge=False,
#     reflection_lm=teacher_lm,
# )

gepa_optimized_program = gepa_optimizer.compile(
    program,
    trainset=trainset,
    valset=valset,
)

result = gepa_optimized_program(question="What is context engineering?")
print(result.answer)

Save the optimized program so you can reload it later without re-running the optimizer.

In [ ]:
gepa_optimized_program.save("gepa_optimized_qa.json")
print("Saved to gepa_optimized_qa.json")